# Branch D — educational trajectory and cultural-capital clustering

This analysis applies the agreed dimension-specific approach to all 191
respondents whose validated final active branch is D. It starts from eight
theory-selected survey variables but performs data-quality screening before
clustering. No composite education or cultural-capital score is constructed.

## Method and theoretical guardrails

- Bourdieu supports parents' education and recognised qualifications as
  indicators of institutionalised/family cultural capital.
- The ILO school-to-work transition framework supports education trajectory,
  expected qualification, work exposure while studying, and re-entry history.
- Original ordinal, nominal, and binary responses are compared with
  equal-weight, missing-aware Gower distance.
- K-medoids and spectral clustering are repeatedly subsampled and combined in
  a consensus ensemble.
- Variables failing transparent coverage or variation checks remain in the
  audit but do not enter the clustering distance.
- Results are exploratory profiles, not validated cultural-capital scores.

In [1]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
VALUES_PATH = ROOT / "outputs/nysc_survey_cleaned_values.csv"
LABELS_PATH = ROOT / "outputs/nysc_survey_cleaned_labels.csv"
OUT = ROOT / "outputs/branch_d_educational_cultural_capital_clusters"
OUT.mkdir(parents=True, exist_ok=True)

THEORY_SELECTED_VARIABLES = [
    "father_edu",
    "mother_edu",
    "edu_status",
    "highest_edu_completed",
    "expected_highest_edu_d",
    "institution_type",
    "worked_while_studying",
    "re_entered_education",
]

CLUSTER_FEATURES = [
    "father_education",
    "mother_education",
    "expected_education",
    "institution_type",
    "worked_while_studying",
    "re_entered_education",
]

RANDOM_STATE = 20260824
MIN_PROFILE_SIZE = 15
CANDIDATE_SUBSAMPLES = 50
CONSENSUS_SUBSAMPLES = 100
SUBSAMPLE_FRACTION = 0.80
ENSEMBLE_BOOTSTRAPS = 100

## 1. Rebuild and audit the strict final-Branch-D cohort

In [2]:
values_all = pd.read_csv(VALUES_PATH)
labels_all = pd.read_csv(LABELS_PATH)

strict_mask = (
    values_all["branch"].eq("D")
    & values_all["branch_routing_source"].eq("translated_final_active_branch")
)

values = values_all.loc[strict_mask, [
    "respondent_id",
    "entry_branch",
    "branch",
    "branch_routing_source",
    "current_activity_status",
] + THEORY_SELECTED_VARIABLES].copy()

labels = labels_all[[
    "respondent_id", "current_activity_status"
] + THEORY_SELECTED_VARIABLES].copy()

cohort = values.merge(
    labels,
    on="respondent_id",
    suffixes=("__value", "__label"),
    validate="one_to_one",
)

validation = {
    "records": int(len(cohort)),
    "unique_ids": int(cohort["respondent_id"].nunique()),
    "wrong_branch": int(cohort["branch"].ne("D").sum()),
    "wrong_routing_source": int(
        cohort["branch_routing_source"].ne("translated_final_active_branch").sum()
    ),
    "not_currently_attending_by_final_status": int(
        cohort["current_activity_status__label"].ne(
            "Attending school, university, or a training programme"
        ).sum()
    ),
}

assert validation["records"] == 191
assert validation["unique_ids"] == 191
assert validation["wrong_branch"] == 0
assert validation["wrong_routing_source"] == 0
assert validation["not_currently_attending_by_final_status"] == 0

display(pd.Series(validation, name="value").to_frame())

,value
records,191
unique_ids,191
wrong_branch,0
wrong_routing_source,0
not_currently_attending_by_final_status,0


## 2. Screen theory-selected variables before clustering

A variable is not retained merely because theory mentions it. It also needs
enough observed data and meaningful variation in this branch.

- `highest_edu_completed` is observed for only 3 respondents, so it cannot
  define distances for the Branch D cohort.
- `edu_status` is dominated by "currently in higher education" and includes
  six answers inconsistent with the final current-activity route. It remains
  available for audit and profile description, but is excluded from distance.

In [3]:
screening_rows = []
for variable in THEORY_SELECTED_VARIABLES:
    value_col = f"{variable}__value"
    label_col = f"{variable}__label"
    observed = cohort[value_col].notna()
    counts = cohort.loc[observed, label_col].value_counts(dropna=False)
    dominant_share = float(counts.iloc[0] / counts.sum()) if len(counts) else np.nan
    screening_rows.append({
        "variable": variable,
        "observed": int(observed.sum()),
        "missing": int((~observed).sum()),
        "coverage_percent": 100 * float(observed.mean()),
        "unique_observed_responses": int(counts.size),
        "dominant_response": str(counts.index[0]) if len(counts) else None,
        "dominant_response_percent": 100 * dominant_share if len(counts) else np.nan,
    })

variable_screening = pd.DataFrame(screening_rows)

status_map = {
    "father_edu": ("include", "family institutionalised cultural-capital indicator"),
    "mother_edu": ("include", "family institutionalised cultural-capital indicator"),
    "edu_status": (
        "exclude_from_distance",
        "over 90% one category and six responses conflict with final current-attendance status",
    ),
    "highest_edu_completed": (
        "exclude_from_distance",
        "observed for only three respondents in this current-student branch",
    ),
    "expected_highest_edu_d": ("include", "expected educational trajectory"),
    "institution_type": ("include", "current institutional position"),
    "worked_while_studying": ("include", "work exposure during education"),
    "re_entered_education": ("include", "interrupted/non-linear education trajectory"),
}

variable_screening["decision"] = variable_screening["variable"].map(
    lambda variable: status_map[variable][0]
)
variable_screening["reason"] = variable_screening["variable"].map(
    lambda variable: status_map[variable][1]
)

contradictory_edu_status = cohort[
    cohort["edu_status__label"].isin([
        "I have completed my education and am not currently studying",
        "I have never studied",
    ])
][[
    "respondent_id",
    "current_activity_status__label",
    "edu_status__label",
    "highest_edu_completed__label",
    "re_entered_education__label",
]]

display(variable_screening.round(2))
display(contradictory_edu_status)

,variable,observed,missing,coverage_percent,unique_observed_responses,dominant_response,dominant_response_percent,decision,reason
0,father_edu,191,0,100.00,8,A/L level (Grade 12–13),29.84,include,family institutionalised cultural-capital indi...
1,mother_edu,191,0,100.00,8,A/L level (Grade 12–13),31.94,include,family institutionalised cultural-capital indi...
2,edu_status,191,0,100.00,6,Currently in higher education (degree or diploma),90.58,exclude_from_distance,over 90% one category and six responses confli...
3,highest_edu_completed,3,188,1.57,1,Bachelor's degree,100.00,exclude_from_distance,observed for only three respondents in this cu...
4,expected_highest_edu_d,191,0,100.00,6,Post-graduate degree,60.73,include,expected educational trajectory
5,institution_type,191,0,100.00,6,Private school or private degree-awarding inst...,57.59,include,current institutional position
6,worked_while_studying,188,3,98.43,2,No,60.64,include,work exposure during education
7,re_entered_education,191,0,100.00,2,No,85.86,include,interrupted/non-linear education trajectory


,respondent_id,current_activity_status__label,edu_status__label,highest_edu_completed__label,re_entered_education__label
23,nysc-8ae4a1d8-97c7-4bc4-9dc9-cb7eea0c897f,"Attending school, university, or a training pr...",I have completed my education and am not curre...,Bachelor's degree,No
59,nysc-4321c45b-467f-41b0-bd23-76872b91a382,"Attending school, university, or a training pr...",I have completed my education and am not curre...,Bachelor's degree,Yes
80,nysc-a3a70af6-24ca-4298-98ca-4390e9366605,"Attending school, university, or a training pr...",I have never studied,NaN,Yes
98,nysc-fc475b43-928f-4e88-a74e-5815c795c629,"Attending school, university, or a training pr...",I have completed my education and am not curre...,Bachelor's degree,No
174,nysc-836e4ae3-c8a3-4aa4-9a9e-a1a3326a9598,"Attending school, university, or a training pr...",I have never studied,NaN,Yes
190,nysc-25c345ec-5f45-4616-adb8-c96e31f3511c,"Attending school, university, or a training pr...",I have never studied,NaN,Yes


## 3. Construct six original-response clustering features

Ordinal variables preserve only their documented survey ordering. Nominal and
binary variables are compared by exact equality. Code 99 for parental
education means "do not know / prefer not to say" and is treated as missing.

In [4]:
features = pd.DataFrame(index=cohort.index)

for source, target in [
    ("father_edu", "father_education"),
    ("mother_edu", "mother_education"),
]:
    raw = pd.to_numeric(cohort[f"{source}__value"], errors="coerce")
    features[target] = raw.where(raw.between(1, 7))

expected = pd.to_numeric(cohort["expected_highest_edu_d__value"], errors="coerce")
features["expected_education"] = expected.where(expected.between(1, 7))

features["institution_type"] = cohort["institution_type__label"].astype("string")
features["worked_while_studying"] = cohort["worked_while_studying__label"].astype("string")
features["re_entered_education"] = cohort["re_entered_education__label"].astype("string")

feature_types = {
    "father_education": "ordinal",
    "mother_education": "ordinal",
    "expected_education": "ordinal",
    "institution_type": "nominal",
    "worked_while_studying": "binary",
    "re_entered_education": "binary",
}

feature_sources = {
    "father_education": "father_edu",
    "mother_education": "mother_edu",
    "expected_education": "expected_highest_edu_d",
    "institution_type": "institution_type",
    "worked_while_studying": "worked_while_studying",
    "re_entered_education": "re_entered_education",
}

feature_audit = pd.DataFrame({
    "feature": CLUSTER_FEATURES,
    "source_variable": [feature_sources[c] for c in CLUSTER_FEATURES],
    "type": [feature_types[c] for c in CLUSTER_FEATURES],
    "observed": [int(features[c].notna().sum()) for c in CLUSTER_FEATURES],
    "missing": [int(features[c].isna().sum()) for c in CLUSTER_FEATURES],
    "maximum_distance_weight": [1 / len(CLUSTER_FEATURES)] * len(CLUSTER_FEATURES),
})

assert features.notna().sum(axis=1).min() >= 4
display(feature_audit)

,feature,source_variable,type,observed,missing,maximum_distance_weight
0,father_education,father_edu,ordinal,186,5,0.166667
1,mother_education,mother_edu,ordinal,183,8,0.166667
2,expected_education,expected_highest_edu_d,ordinal,191,0,0.166667
3,institution_type,institution_type,nominal,191,0,0.166667
4,worked_while_studying,worked_while_studying,binary,188,3,0.166667
5,re_entered_education,re_entered_education,binary,191,0,0.166667


## 4. Calculate equal-weight, missing-aware Gower distance

In [5]:
def mixed_gower(frame, columns=None):
    if columns is None:
        columns = list(frame.columns)
    n = len(frame)
    numerator = np.zeros((n, n), dtype=float)
    denominator = np.zeros((n, n), dtype=float)

    for column in columns:
        if feature_types[column] == "ordinal":
            values_k = pd.to_numeric(frame[column], errors="coerce").to_numpy(float)
            observed = np.isfinite(values_k)
            pair_observed = observed[:, None] & observed[None, :]
            difference = np.abs(values_k[:, None] - values_k[None, :]) / 6.0
        else:
            string_values = frame[column].astype("string")
            observed = string_values.notna().to_numpy()
            values_k = string_values.fillna("__MISSING__").to_numpy(dtype=str)
            pair_observed = observed[:, None] & observed[None, :]
            difference = (values_k[:, None] != values_k[None, :]).astype(float)

        numerator += np.where(pair_observed, difference, 0.0)
        denominator += pair_observed

    distance = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0,
    )
    np.fill_diagonal(distance, 0.0)
    return distance, denominator


D, pair_feature_coverage = mixed_gower(features)
assert np.allclose(D, D.T)
assert D.min() >= 0 and D.max() <= 1
assert pair_feature_coverage.min() >= 3

positive_distances = D[D > 0]
sigma = float(np.median(positive_distances))


def affinity(distance):
    result = np.exp(-(distance ** 2) / (2 * sigma ** 2))
    np.fill_diagonal(result, 1.0)
    return result


print("Gower distance median:", round(float(np.median(positive_distances)), 4))
print(
    "Pairwise observed features: min",
    int(pair_feature_coverage.min()),
    "median",
    int(np.median(pair_feature_coverage)),
    "max",
    int(pair_feature_coverage.max()),
)

Gower distance median: 0.3333
Pairwise observed features: min 3 median 6 max 6


## 5. Compare candidate profile counts

Candidate `k=2–5` solutions are compared using separation, 80%-subsample
stability, cross-algorithm agreement, and the minimum profile-size rule.

In [6]:
def fit_kmedoids(distance, k, n_init=200, max_iter=200, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = len(distance)
    best = None
    for _ in range(n_init):
        medoids = np.sort(rng.choice(n, size=k, replace=False))
        for _ in range(max_iter):
            labels_k = np.argmin(distance[:, medoids], axis=1)
            updated = medoids.copy()
            for cluster in range(k):
                members = np.flatnonzero(labels_k == cluster)
                if len(members) == 0:
                    nearest = distance[:, medoids].min(axis=1)
                    available = np.setdiff1d(np.argsort(nearest)[::-1], updated)
                    updated[cluster] = available[0]
                else:
                    within = distance[np.ix_(members, members)]
                    updated[cluster] = members[np.argmin(within.sum(axis=1))]
            if np.array_equal(updated, medoids):
                break
            medoids = updated
        labels_k = np.argmin(distance[:, medoids], axis=1)
        loss = float(distance[np.arange(n), medoids[labels_k]].sum())
        candidate = (loss, tuple(medoids.tolist()), medoids.copy(), labels_k.copy())
        if best is None or candidate[:2] < best[:2]:
            best = candidate
    return {"labels": best[3], "medoids": best[2], "loss": best[0]}


def medoids_from_labels(distance, labels_k):
    result = []
    for cluster in sorted(np.unique(labels_k)):
        members = np.flatnonzero(labels_k == cluster)
        within = distance[np.ix_(members, members)]
        result.append(members[np.argmin(within.sum(axis=1))])
    return np.asarray(result, dtype=int)


def subsample_stability(algorithm, k, reference_labels):
    offset = 0 if algorithm == "kmedoids" else 10000
    rng = np.random.default_rng(RANDOM_STATE + 100 * k + offset)
    rows = []
    sample_size = int(np.floor(SUBSAMPLE_FRACTION * len(D)))
    for repetition in range(CANDIDATE_SUBSAMPLES):
        idx = np.sort(rng.choice(len(D), size=sample_size, replace=False))
        D_sub = D[np.ix_(idx, idx)]
        if algorithm == "kmedoids":
            result = fit_kmedoids(
                D_sub,
                k,
                n_init=50,
                random_state=RANDOM_STATE + 1000 * k + repetition,
            )
            medoids = idx[result["medoids"]]
        else:
            labels_sub = SpectralClustering(
                n_clusters=k,
                affinity="precomputed",
                assign_labels="kmeans",
                n_init=30,
                random_state=RANDOM_STATE + 2000 * k + repetition,
            ).fit_predict(affinity(D_sub))
            medoids = idx[medoids_from_labels(D_sub, labels_sub)]
        predicted = np.argmin(D[:, medoids], axis=1)
        rows.append(adjusted_rand_score(reference_labels, predicted))
    return np.asarray(rows)


candidate_rows = []
candidate_stability_rows = []
full_source_solutions = {}

for k in range(2, 6):
    kmedoids = fit_kmedoids(D, k, n_init=400, random_state=RANDOM_STATE + k)
    spectral_labels = SpectralClustering(
        n_clusters=k,
        affinity="precomputed",
        assign_labels="kmeans",
        n_init=100,
        random_state=RANDOM_STATE + 100 + k,
    ).fit_predict(affinity(D))

    full_source_solutions[("kmedoids", k)] = kmedoids["labels"]
    full_source_solutions[("spectral", k)] = spectral_labels
    method_values = {}

    for algorithm, labels_k in [
        ("kmedoids", kmedoids["labels"]),
        ("spectral", spectral_labels),
    ]:
        stability = subsample_stability(algorithm, k, labels_k)
        method_values[algorithm] = {
            "silhouette": silhouette_score(D, labels_k, metric="precomputed"),
            "stability": float(stability.mean()),
            "stability_p10": float(np.quantile(stability, 0.10)),
            "minimum_size": int(pd.Series(labels_k).value_counts().min()),
        }
        candidate_stability_rows.extend({
            "algorithm": algorithm,
            "k": k,
            "repetition": repetition + 1,
            "ari": value,
        } for repetition, value in enumerate(stability))

    candidate_rows.append({
        "k": k,
        "kmedoids_silhouette": method_values["kmedoids"]["silhouette"],
        "spectral_silhouette": method_values["spectral"]["silhouette"],
        "mean_source_silhouette": np.mean([
            method_values["kmedoids"]["silhouette"],
            method_values["spectral"]["silhouette"],
        ]),
        "kmedoids_stability_mean": method_values["kmedoids"]["stability"],
        "spectral_stability_mean": method_values["spectral"]["stability"],
        "mean_source_stability": np.mean([
            method_values["kmedoids"]["stability"],
            method_values["spectral"]["stability"],
        ]),
        "minimum_source_p10_stability": min(
            method_values["kmedoids"]["stability_p10"],
            method_values["spectral"]["stability_p10"],
        ),
        "source_algorithm_ari": adjusted_rand_score(
            kmedoids["labels"], spectral_labels
        ),
        "minimum_cluster_size_across_sources": min(
            method_values["kmedoids"]["minimum_size"],
            method_values["spectral"]["minimum_size"],
        ),
    })

candidate_metrics = pd.DataFrame(candidate_rows)
candidate_stability = pd.DataFrame(candidate_stability_rows)

eligible = candidate_metrics.loc[
    candidate_metrics["minimum_cluster_size_across_sources"].ge(MIN_PROFILE_SIZE)
].copy()
if eligible.empty:
    raise RuntimeError("No k=2–5 candidate meets the minimum profile-size rule.")

eligible["rank_mean_silhouette"] = eligible["mean_source_silhouette"].rank(
    ascending=False, method="min"
)
eligible["rank_mean_stability"] = eligible["mean_source_stability"].rank(
    ascending=False, method="min"
)
eligible["primary_mean_rank"] = eligible[[
    "rank_mean_silhouette", "rank_mean_stability"
]].mean(axis=1)

best_rank = eligible["primary_mean_rank"].min()
selected_k = int(
    eligible.loc[eligible["primary_mean_rank"].eq(best_rank)]
    .sort_values("k")
    .iloc[0]["k"]
)

candidate_metrics = candidate_metrics.merge(
    eligible[["k", "primary_mean_rank"]], on="k", how="left"
)
candidate_metrics["selected"] = candidate_metrics["k"].eq(selected_k)

display(candidate_metrics.round(4))
print("Selected common k:", selected_k)

,k,kmedoids_silhouette,spectral_silhouette,mean_source_silhouette,kmedoids_stability_mean,spectral_stability_mean,mean_source_stability,minimum_source_p10_stability,source_algorithm_ari,minimum_cluster_size_across_sources,primary_mean_rank,selected
0,2,0.4114,0.3834,0.3974,0.7302,0.7070,0.7186,0.2856,0.8383,69,3.0,False
1,3,0.3909,0.4045,0.3977,0.9237,0.6001,0.7619,0.3325,0.7544,52,2.0,False
2,4,0.4128,0.4227,0.4177,0.9104,0.7461,0.8282,0.7147,0.7147,15,1.0,True
3,5,0.4411,0.4315,0.4363,0.8829,0.8107,0.8468,0.6763,0.8067,13,NaN,False


Selected common k: 4


## 6. Build the equal-weight K-medoids + spectral consensus ensemble

In [7]:
ensemble_rng = np.random.default_rng(RANDOM_STATE + 30000)
n = len(D)
sample_size = int(np.floor(SUBSAMPLE_FRACTION * n))
cover_matrices = []
same_matrices = []
base_rows = []

for repetition in range(CONSENSUS_SUBSAMPLES):
    idx = np.sort(ensemble_rng.choice(n, size=sample_size, replace=False))
    D_sub = D[np.ix_(idx, idx)]
    cover = np.zeros((n, n), dtype=np.uint8)
    cover[np.ix_(idx, idx)] = 1

    kmedoids_sub = fit_kmedoids(
        D_sub,
        selected_k,
        n_init=100,
        random_state=RANDOM_STATE + 40000 + repetition,
    )["labels"]

    spectral_sub = SpectralClustering(
        n_clusters=selected_k,
        affinity="precomputed",
        assign_labels="kmeans",
        n_init=30,
        random_state=RANDOM_STATE + 50000 + repetition,
    ).fit_predict(affinity(D_sub))

    for algorithm, labels_sub in [
        ("kmedoids", kmedoids_sub),
        ("spectral", spectral_sub),
    ]:
        same = np.zeros((n, n), dtype=np.uint8)
        same[np.ix_(idx, idx)] = (
            labels_sub[:, None] == labels_sub[None, :]
        ).astype(np.uint8)
        cover_matrices.append(cover.copy())
        same_matrices.append(same)
        base_rows.append({
            "partition_id": len(base_rows) + 1,
            "algorithm": algorithm,
            "repetition": repetition + 1,
            "silhouette_in_subsample": silhouette_score(
                D_sub, labels_sub, metric="precomputed"
            ),
            "minimum_cluster_size": int(pd.Series(labels_sub).value_counts().min()),
        })

cover_stack = np.stack(cover_matrices)
same_stack = np.stack(same_matrices)
coverage = cover_stack.sum(axis=0)
together = same_stack.sum(axis=0)
consensus = together / coverage
np.fill_diagonal(consensus, 1.0)
consensus_distance = 1 - consensus
np.fill_diagonal(consensus_distance, 0.0)

base_partition_diagnostics = pd.DataFrame(base_rows)
final_fit = fit_kmedoids(
    consensus_distance,
    selected_k,
    n_init=500,
    random_state=RANDOM_STATE + 60000,
)
raw_labels = final_fit["labels"]

# Stable, neutral IDs ordered by medoid respondent identifier; no high/low score.
raw_medoids = {}
for raw_cluster in sorted(np.unique(raw_labels)):
    members = np.flatnonzero(raw_labels == raw_cluster)
    within = consensus_distance[np.ix_(members, members)]
    medoid_idx = members[np.argmin(within.sum(axis=1))]
    raw_medoids[int(raw_cluster)] = medoid_idx

raw_order = sorted(
    raw_medoids,
    key=lambda label: str(cohort.iloc[raw_medoids[label]]["respondent_id"]),
)
raw_to_profile = {raw: profile + 1 for profile, raw in enumerate(raw_order)}
profile_id = np.asarray([raw_to_profile[int(label)] for label in raw_labels])
if selected_k == 4:
    # Descriptive names assigned after inspecting the complete response modes.
    # They describe dominant patterns and do not rank respondents.
    profile_names = {
        1: "Re-entered students, often with degree-educated parents",
        2: "State-institution students without work exposure",
        3: "Private-institution students without work exposure",
        4: "Work-experienced students, mainly in private institutions",
    }
else:
    profile_names = {
        profile: f"Educational/cultural profile {profile}"
        for profile in sorted(np.unique(profile_id))
    }

print("Base partitions:", len(base_partition_diagnostics))
print(
    "Pair coverage: min",
    int(coverage.min()),
    "median",
    int(np.median(coverage)),
    "max",
    int(coverage.max()),
)

Base partitions: 200
Pair coverage: min 90 median 126 max 180


## 7. Profile clusters using original readable responses

Profiles are numbered neutrally. Interpretation comes from medians, modes, and
complete within-profile response distributions—not from an invented score.

In [8]:
assignments = cohort[[
    "respondent_id", "entry_branch", "branch", "branch_routing_source"
]].copy()

for variable in THEORY_SELECTED_VARIABLES:
    assignments[f"{variable}__value"] = cohort[f"{variable}__value"]
    assignments[f"{variable}__response"] = cohort[f"{variable}__label"]

for feature in CLUSTER_FEATURES:
    assignments[feature] = features[feature]

assignments["educational_cultural_profile_id"] = profile_id
assignments["educational_cultural_profile_label"] = assignments[
    "educational_cultural_profile_id"
].map(profile_names)

gower_individual_silhouette = silhouette_samples(
    D, profile_id, metric="precomputed"
)
consensus_individual_silhouette = silhouette_samples(
    consensus_distance, profile_id, metric="precomputed"
)
assignments["gower_individual_silhouette"] = gower_individual_silhouette
assignments["consensus_individual_silhouette"] = consensus_individual_silhouette

profile_sizes = (
    assignments.groupby([
        "educational_cultural_profile_id",
        "educational_cultural_profile_label",
    ], as_index=False)
    .agg(
        records=("respondent_id", "size"),
        low_gower_confidence_records=(
            "gower_individual_silhouette", lambda values: int((values < 0.10).sum())
        ),
        mean_gower_silhouette=("gower_individual_silhouette", "mean"),
        mean_consensus_silhouette=("consensus_individual_silhouette", "mean"),
    )
)
profile_sizes["percent"] = 100 * profile_sizes["records"] / len(assignments)

ordinal_profile_medians = pd.DataFrame({
    "father_education_median": assignments.groupby(
        "educational_cultural_profile_id"
    )["father_education"].median(),
    "mother_education_median": assignments.groupby(
        "educational_cultural_profile_id"
    )["mother_education"].median(),
    "expected_education_median": assignments.groupby(
        "educational_cultural_profile_id"
    )["expected_education"].median(),
})

response_rows = []
for feature in CLUSTER_FEATURES:
    source = feature_sources[feature]
    response_col = f"{source}__response"
    temp = (
        assignments.groupby(
            ["educational_cultural_profile_id", response_col],
            dropna=False,
        )
        .size()
        .rename("records")
        .reset_index()
    )
    temp["within_profile_percent"] = (
        100
        * temp["records"]
        / temp.groupby("educational_cultural_profile_id")["records"].transform("sum")
    )
    temp = temp.rename(columns={response_col: "response"})
    temp.insert(1, "feature", feature)
    temp.insert(2, "source_variable", source)
    response_rows.append(temp)

profile_response_distributions = pd.concat(response_rows, ignore_index=True)
profile_modes = (
    profile_response_distributions.sort_values(
        ["educational_cultural_profile_id", "feature", "within_profile_percent"],
        ascending=[True, True, False],
    )
    .groupby(["educational_cultural_profile_id", "feature"], as_index=False)
    .first()
)

medoid_rows = []
for raw_cluster, medoid_idx in raw_medoids.items():
    pid = raw_to_profile[raw_cluster]
    row = {
        "educational_cultural_profile_id": pid,
        "respondent_id": cohort.iloc[medoid_idx]["respondent_id"],
    }
    for feature in CLUSTER_FEATURES:
        source = feature_sources[feature]
        row[f"{feature}__response"] = cohort.iloc[medoid_idx][f"{source}__label"]
    medoid_rows.append(row)
medoid_profiles = pd.DataFrame(medoid_rows).sort_values(
    "educational_cultural_profile_id"
)

display(profile_sizes.round(3))
display(ordinal_profile_medians)
display(profile_modes.round(2))
display(medoid_profiles)

,educational_cultural_profile_id,educational_cultural_profile_label,records,low_gower_confidence_records,mean_gower_silhouette,mean_consensus_silhouette,percent
0,1,"Re-entered students, often with degree-educate...",15,4,0.262,0.739,7.853
1,2,State-institution students without work exposure,53,8,0.393,0.843,27.749
2,3,Private-institution students without work expo...,57,1,0.543,0.879,29.843
3,4,"Work-experienced students, mainly in private i...",66,2,0.391,0.826,34.555


,father_education_median,mother_education_median,expected_education_median
educational_cultural_profile_id,,,
1,6.0,6.0,6.0
2,3.5,3.0,7.0
3,3.0,5.0,7.0
4,4.0,5.0,7.0


,educational_cultural_profile_id,feature,source_variable,response,records,within_profile_percent
0,1,expected_education,expected_highest_edu_d,Bachelor's degree,8,53.33
1,1,father_education,father_edu,Bachelor's degree,12,80.00
2,1,institution_type,institution_type,Private school or private degree-awarding inst...,7,46.67
3,1,mother_education,mother_edu,Bachelor's degree,11,73.33
4,1,re_entered_education,re_entered_education,Yes,15,100.00
5,1,worked_while_studying,worked_while_studying,Yes,8,53.33
6,2,expected_education,expected_highest_edu_d,Post-graduate degree,33,62.26
7,2,father_education,father_edu,Bachelor's degree,14,26.42
8,2,institution_type,institution_type,Government school or state university,44,83.02
9,2,mother_education,mother_edu,A/L level (Grade 12–13),22,41.51


,educational_cultural_profile_id,respondent_id,father_education__response,mother_education__response,expected_education__response,institution_type__response,worked_while_studying__response,re_entered_education__response
1,1,nysc-a3a70af6-24ca-4298-98ca-4390e9366605,Bachelor's degree,Bachelor's degree,Bachelor's degree,Private school or private degree-awarding inst...,NaN,Yes
2,2,nysc-bb28e1a6-f66a-4ced-8bf4-e88fa2012208,Diploma,O/L level (Grade 9–11),Post-graduate degree,Government school or state university,No,No
3,3,nysc-e49f1cfe-9cf3-4091-8c51-01c2913290b3,Bachelor's degree,Bachelor's degree,Post-graduate degree,Private school or private degree-awarding inst...,No,No
0,4,nysc-f73ce24e-0e7e-4640-a105-f70e73989176,A/L level (Grade 12–13),Diploma,Post-graduate degree,Private school or private degree-awarding inst...,Yes,No


## 8. Validate stability, ambiguity, and sensitivity to each feature

In [9]:
confidence_rows = []
for i in range(n):
    own = profile_id[i]
    own_members = np.flatnonzero(profile_id == own)
    own_members = own_members[own_members != i]
    within = float(consensus[i, own_members].mean()) if len(own_members) else 1.0
    strongest_other = max(
        float(consensus[i, np.flatnonzero(profile_id == other)].mean())
        for other in sorted(set(profile_id) - {own})
    )
    confidence_rows.append({
        "respondent_id": cohort.iloc[i]["respondent_id"],
        "educational_cultural_profile_id": int(own),
        "within_profile_consensus": within,
        "strongest_other_profile_consensus": strongest_other,
        "consensus_margin": within - strongest_other,
        "ambiguous_consensus_membership": (
            (within < 0.70) or ((within - strongest_other) < 0.20)
        ),
    })

respondent_confidence = pd.DataFrame(confidence_rows)
assignments = assignments.merge(
    respondent_confidence.drop(columns="educational_cultural_profile_id"),
    on="respondent_id",
    validate="one_to_one",
)

bootstrap_rng = np.random.default_rng(RANDOM_STATE + 70000)
ensemble_stability_rows = []
for repetition in range(ENSEMBLE_BOOTSTRAPS):
    selected = bootstrap_rng.choice(
        len(cover_stack), size=len(cover_stack), replace=True
    )
    boot_cover = cover_stack[selected].sum(axis=0)
    boot_same = same_stack[selected].sum(axis=0)
    boot_consensus = np.divide(
        boot_same,
        boot_cover,
        out=np.zeros_like(boot_same, dtype=float),
        where=boot_cover > 0,
    )
    np.fill_diagonal(boot_consensus, 1.0)
    boot_distance = 1 - boot_consensus
    np.fill_diagonal(boot_distance, 0.0)
    boot_labels = fit_kmedoids(
        boot_distance,
        selected_k,
        n_init=80,
        random_state=RANDOM_STATE + 80000 + repetition,
    )["labels"]
    ensemble_stability_rows.append({
        "repetition": repetition + 1,
        "ari_vs_full_consensus": adjusted_rand_score(profile_id, boot_labels),
    })

ensemble_stability = pd.DataFrame(ensemble_stability_rows)
full_kmedoids = full_source_solutions[("kmedoids", selected_k)]
full_spectral = full_source_solutions[("spectral", selected_k)]
consensus_hac_average = AgglomerativeClustering(
    n_clusters=selected_k, metric="precomputed", linkage="average"
).fit_predict(consensus_distance)
consensus_hac_complete = AgglomerativeClustering(
    n_clusters=selected_k, metric="precomputed", linkage="complete"
).fit_predict(consensus_distance)

upper = consensus[np.triu_indices(n, 1)]
pac_10_90 = float(np.mean((upper > 0.10) & (upper < 0.90)))
metrics = {
    "gower_silhouette": float(silhouette_score(D, profile_id, metric="precomputed")),
    "consensus_distance_silhouette": float(
        silhouette_score(consensus_distance, profile_id, metric="precomputed")
    ),
    "ensemble_bootstrap_ari_mean": float(
        ensemble_stability["ari_vs_full_consensus"].mean()
    ),
    "ensemble_bootstrap_ari_p10": float(
        ensemble_stability["ari_vs_full_consensus"].quantile(0.10)
    ),
    "ari_vs_full_kmedoids": float(adjusted_rand_score(profile_id, full_kmedoids)),
    "ari_vs_full_spectral": float(adjusted_rand_score(profile_id, full_spectral)),
    "ari_consensus_kmedoids_vs_hac_average": float(
        adjusted_rand_score(profile_id, consensus_hac_average)
    ),
    "ari_consensus_kmedoids_vs_hac_complete": float(
        adjusted_rand_score(profile_id, consensus_hac_complete)
    ),
    "minimum_profile_size": int(pd.Series(profile_id).value_counts().min()),
    "low_gower_confidence_percent": 100 * float(
        np.mean(gower_individual_silhouette < 0.10)
    ),
    "ambiguous_consensus_percent": 100 * float(
        assignments["ambiguous_consensus_membership"].mean()
    ),
    "pac_10_90": pac_10_90,
}

feature_sensitivity_rows = []
for omitted in CLUSTER_FEATURES:
    retained = [feature for feature in CLUSTER_FEATURES if feature != omitted]
    D_reduced, coverage_reduced = mixed_gower(features, retained)
    reduced_fit = fit_kmedoids(
        D_reduced,
        selected_k,
        n_init=400,
        random_state=RANDOM_STATE + 90000 + CLUSTER_FEATURES.index(omitted),
    )
    feature_sensitivity_rows.append({
        "omitted_feature": omitted,
        "retained_features": len(retained),
        "minimum_pair_coverage": int(coverage_reduced.min()),
        "gower_silhouette_reduced_kmedoids": float(
            silhouette_score(D_reduced, reduced_fit["labels"], metric="precomputed")
        ),
        "ari_reduced_kmedoids_vs_final_consensus": float(
            adjusted_rand_score(profile_id, reduced_fit["labels"])
        ),
    })
feature_sensitivity = pd.DataFrame(feature_sensitivity_rows)

quality_checks = pd.DataFrame([
    {
        "check": "original Gower silhouette >= 0.25",
        "value": metrics["gower_silhouette"],
        "passed": metrics["gower_silhouette"] >= 0.25,
    },
    {
        "check": "ensemble bootstrap mean ARI >= 0.80",
        "value": metrics["ensemble_bootstrap_ari_mean"],
        "passed": metrics["ensemble_bootstrap_ari_mean"] >= 0.80,
    },
    {
        "check": f"minimum profile size >= {MIN_PROFILE_SIZE}",
        "value": metrics["minimum_profile_size"],
        "passed": metrics["minimum_profile_size"] >= MIN_PROFILE_SIZE,
    },
    {
        "check": "ARI with full K-medoids >= 0.60",
        "value": metrics["ari_vs_full_kmedoids"],
        "passed": metrics["ari_vs_full_kmedoids"] >= 0.60,
    },
    {
        "check": "ARI with full spectral >= 0.60",
        "value": metrics["ari_vs_full_spectral"],
        "passed": metrics["ari_vs_full_spectral"] >= 0.60,
    },
    {
        "check": "PAC (0.10, 0.90) <= 0.20",
        "value": metrics["pac_10_90"],
        "passed": metrics["pac_10_90"] <= 0.20,
    },
    {
        "check": "ambiguous consensus respondents <= 20%",
        "value": metrics["ambiguous_consensus_percent"],
        "passed": metrics["ambiguous_consensus_percent"] <= 20,
    },
    {
        "check": "low-confidence respondents in original Gower space <= 20%",
        "value": metrics["low_gower_confidence_percent"],
        "passed": metrics["low_gower_confidence_percent"] <= 20,
    },
])

primary_pass = bool(quality_checks.iloc[:5]["passed"].all())
if primary_pass and quality_checks.iloc[5:]["passed"].all():
    status = "Stable, cross-method-supported educational/cultural segmentation"
elif primary_pass:
    status = "Passes primary stability rules but retains ambiguity cautions"
else:
    status = "No sufficiently robust educational/cultural segmentation found"

display(pd.Series(metrics, name="value").to_frame().round(3))
display(quality_checks.round(3))
display(feature_sensitivity.round(3))
print("Decision:", status)

,value
gower_silhouette,0.427
consensus_distance_silhouette,0.840
ensemble_bootstrap_ari_mean,0.963
ensemble_bootstrap_ari_p10,0.917
ari_vs_full_kmedoids,0.776
ari_vs_full_spectral,0.934
ari_consensus_kmedoids_vs_hac_average,0.917
ari_consensus_kmedoids_vs_hac_complete,0.917
minimum_profile_size,15.000
low_gower_confidence_percent,7.853


,check,value,passed
0,original Gower silhouette >= 0.25,0.427,True
1,ensemble bootstrap mean ARI >= 0.80,0.963,True
2,minimum profile size >= 15,15.000,True
3,ARI with full K-medoids >= 0.60,0.776,True
4,ARI with full spectral >= 0.60,0.934,True
5,"PAC (0.10, 0.90) <= 0.20",0.134,True
6,ambiguous consensus respondents <= 20%,12.565,True
7,low-confidence respondents in original Gower s...,7.853,True


,omitted_feature,retained_features,minimum_pair_coverage,gower_silhouette_reduced_kmedoids,ari_reduced_kmedoids_vs_final_consensus
0,father_education,5,3,0.517,0.817
1,mother_education,5,3,0.481,0.870
2,expected_education,5,2,0.466,0.808
3,institution_type,5,2,0.370,0.546
4,worked_while_studying,5,3,0.492,0.383
5,re_entered_education,5,2,0.467,0.720


Decision: Stable, cross-method-supported educational/cultural segmentation


## 9. Visualise candidate evidence, consensus, and profiles

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(
    candidate_metrics["k"],
    candidate_metrics["mean_source_silhouette"],
    marker="o",
    label="mean source silhouette",
)
axes[0].axvline(selected_k, color="tab:red", linestyle="--", label=f"selected k={selected_k}")
axes[0].axhline(0.25, color="grey", linestyle=":")
axes[0].set(
    title="Candidate separation",
    xlabel="Number of profiles",
    ylabel="Mean Gower silhouette",
)
axes[0].legend()

axes[1].plot(
    candidate_metrics["k"],
    candidate_metrics["mean_source_stability"],
    marker="o",
    label="mean source stability",
)
axes[1].plot(
    candidate_metrics["k"],
    candidate_metrics["source_algorithm_ari"],
    marker="s",
    label="K-medoids vs spectral ARI",
)
axes[1].axvline(selected_k, color="tab:red", linestyle="--")
axes[1].axhline(0.70, color="grey", linestyle=":")
axes[1].set(
    title="Stability and source agreement",
    xlabel="Number of profiles",
    ylabel="ARI",
    ylim=(-0.05, 1.05),
)
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "educational_cultural_candidate_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()

order = np.argsort(profile_id, kind="stable")
fig, ax = plt.subplots(figsize=(8, 7))
heat = ax.imshow(
    consensus[np.ix_(order, order)],
    cmap="viridis",
    vmin=0,
    vmax=1,
    aspect="auto",
)
ordered_counts = pd.Series(profile_id[order]).value_counts(sort=False).sort_index().to_numpy()
boundaries = np.cumsum(ordered_counts)[:-1]
for boundary in boundaries:
    ax.axhline(boundary - 0.5, color="white", linewidth=1.2)
    ax.axvline(boundary - 0.5, color="white", linewidth=1.2)
ax.set(
    title="Educational/cultural consensus matrix",
    xlabel="Respondents ordered by profile",
    ylabel="Respondents",
)
fig.colorbar(heat, ax=ax, label="Proportion clustered together")
fig.tight_layout()
fig.savefig(OUT / "educational_cultural_consensus_matrix_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()

# Classical metric MDS is used only to visualise the Gower-distance structure.
J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (D ** 2) @ J
eigenvalues, eigenvectors = np.linalg.eigh(B)
order_eigen = np.argsort(eigenvalues)[::-1]
positive = order_eigen[eigenvalues[order_eigen] > 0][:2]
coordinates = eigenvectors[:, positive] * np.sqrt(eigenvalues[positive])

fig, ax = plt.subplots(figsize=(8, 6))
for pid in sorted(np.unique(profile_id)):
    mask = profile_id == pid
    ax.scatter(
        coordinates[mask, 0],
        coordinates[mask, 1],
        s=40,
        alpha=0.75,
        label=profile_names[int(pid)],
    )
ax.set(
    title="Educational/cultural profiles in two-dimensional Gower space",
    xlabel="Classical MDS axis 1",
    ylabel="Classical MDS axis 2",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "educational_cultural_profile_scatterplot.png", dpi=180, bbox_inches="tight")
plt.show()

/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2298/2928944726.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2298/2928944726.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2298/2928944726.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Save reproducible outputs

In [11]:
variable_screening.to_csv(OUT / "educational_cultural_variable_screening.csv", index=False)
contradictory_edu_status.to_csv(OUT / "educational_cultural_edu_status_conflicts.csv", index=False)
feature_audit.to_csv(OUT / "educational_cultural_feature_audit.csv", index=False)
candidate_metrics.to_csv(OUT / "educational_cultural_candidate_metrics.csv", index=False)
candidate_stability.to_csv(OUT / "educational_cultural_candidate_subsample_stability.csv", index=False)
base_partition_diagnostics.to_csv(OUT / "educational_cultural_consensus_base_partitions.csv", index=False)
assignments.to_csv(OUT / "educational_cultural_consensus_assignments.csv", index=False)
profile_sizes.to_csv(OUT / "educational_cultural_profile_sizes.csv", index=False)
ordinal_profile_medians.to_csv(OUT / "educational_cultural_profile_ordinal_medians.csv")
profile_response_distributions.to_csv(OUT / "educational_cultural_profile_response_distributions.csv", index=False)
profile_modes.to_csv(OUT / "educational_cultural_profile_modes.csv", index=False)
medoid_profiles.to_csv(OUT / "educational_cultural_medoid_profiles.csv", index=False)
respondent_confidence.to_csv(OUT / "educational_cultural_respondent_confidence.csv", index=False)
ensemble_stability.to_csv(OUT / "educational_cultural_consensus_ensemble_stability.csv", index=False)
feature_sensitivity.to_csv(OUT / "educational_cultural_leave_one_feature_out.csv", index=False)
quality_checks.to_csv(OUT / "educational_cultural_quality_checks.csv", index=False)

consensus_export = pd.DataFrame(consensus, columns=cohort["respondent_id"])
consensus_export.insert(0, "respondent_id", cohort["respondent_id"].to_numpy())
consensus_export.to_csv(OUT / "educational_cultural_consensus_matrix.csv", index=False)

summary = {
    "cohort": {
        "records": int(len(cohort)),
        "branch": "D",
        "routing_source": "translated_final_active_branch",
    },
    "dimension": "educational trajectory and cultural capital",
    "theory_selected_variables": THEORY_SELECTED_VARIABLES,
    "excluded_from_distance_after_screening": {
        "edu_status": status_map["edu_status"][1],
        "highest_edu_completed": status_map["highest_edu_completed"][1],
    },
    "clustering_features": CLUSTER_FEATURES,
    "data_quality": validation,
    "method": {
        "distance": "equal-weight mixed, missing-aware Gower on original responses",
        "candidate_k": [2, 3, 4, 5],
        "selected_k": selected_k,
        "ensemble": "equal-weight K-medoids plus spectral consensus",
        "base_partitions": 2 * CONSENSUS_SUBSAMPLES,
        "final_method": "K-medoids on one minus co-association",
    },
    "profiles": {
        "sizes": {
            str(int(row.educational_cultural_profile_id)): int(row.records)
            for row in profile_sizes.itertuples()
        },
        "names": {str(key): value for key, value in profile_names.items()},
    },
    "metrics": metrics,
    "decision": {
        "primary_checks_passed": primary_pass,
        "checks_passed": int(quality_checks["passed"].sum()),
        "checks_total": int(len(quality_checks)),
        "status": status,
    },
    "limitations": [
        "The profiles are exploratory and are not validated cultural-capital scores.",
        "Parental education is an indicator/proxy; it does not fully measure family cultural capital.",
        "Highest completed education is unavailable for 188 of 191 respondents and is excluded from distance.",
        "Education status is near-constant and contains six conflicts with final current-attendance status, so it is excluded from distance.",
        "Institution type, work exposure, and re-entry are compared as categories without invented ranks.",
        "Leave-one-feature-out results show greatest sensitivity to work-while-studying and institution type; profile interpretation must foreground those drivers.",
    ],
}

with open(OUT / "educational_cultural_clustering_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved files:")
for path in sorted(OUT.iterdir()):
    print(" -", path.name)

{
  "cohort": {
    "records": 191,
    "branch": "D",
    "routing_source": "translated_final_active_branch"
  },
  "dimension": "educational trajectory and cultural capital",
  "theory_selected_variables": [
    "father_edu",
    "mother_edu",
    "edu_status",
    "highest_edu_completed",
    "expected_highest_edu_d",
    "institution_type",
    "worked_while_studying",
    "re_entered_education"
  ],
  "excluded_from_distance_after_screening": {
    "edu_status": "over 90% one category and six responses conflict with final current-attendance status",
    "highest_edu_completed": "observed for only three respondents in this current-student branch"
  },
  "clustering_features": [
    "father_education",
    "mother_education",
    "expected_education",
    "institution_type",
    "worked_while_studying",
    "re_entered_education"
  ],
  "data_quality": {
    "records": 191,
    "unique_ids": 191,
    "wrong_branch": 0,
    "wrong_routing_source": 0,
    "not_currently_attending_by_f

## Interpretation guardrails

- These profiles describe similarities in family educational background,
  expected qualification, institution type, work exposure while studying, and
  education re-entry history.
- They do not measure intelligence, merit, programme suitability, respondent
  worth, or causal effects of parents' education.
- A stable consensus indicates reproducible grouping under the selected inputs;
  it does not validate a cultural-capital scale.
- Profile interpretation must use the full original-response distributions and
  leave-one-feature-out sensitivity results.